In [ ]:
'''
problem statement : To predict loan aproval yes / no
------------------------------------------------------------------------------------------------------------------------------------------------------------------
workflow:
        input feature (X):
        target columns(y):

step1: data ingestion ( data load)
step2: data_exploration ( descritive stats)
step3: data_preprocessing 
    1) drop the duplicates
    2) split the X and y i.e input feature / target 
    3) train_test_split
    4) use the scalling ( pipeline)
    5) use the SMOOTE ( if applicable)
step4: data_building
step5: model_evaluation       
'''  


# import the manipulation labraries
import pandas as pd 
import numpy as np 

#import the visualition labraries
import matplotlib.pyplot as plt 
import seaborn as sns

# import filter warnings
import warnings
warnings.filterwarnings(action="ignore")

# import scikit-learn labraries for preprocessing

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler , RobustScaler , LabelEncoder, OneHotEncoder
from sklearn.metrics import accuracy_score , precision_score , f1_score, recall_score , classification_report
from sklearn.impute import SimpleImputer
from imblearn.over_sampling import SMOTE
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from collections import OrderedDict


#import scikit-learn labraries for the model_building

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier , AdaBoostClassifier , GradientBoostingClassifier , ExtraTreesClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier

# step1: data_ingestion 
df =pd.read_csv('D:\Loan_aproval_predictionModelClassification\data\loan_approval_dataset.csv')
df.columns



#step2 : data_explaration 

numerical_col = df.select_dtypes(exclude='object').columns
categorical_col = df.select_dtypes(include='object').columns

Q1 = df[numerical_col].quantile(0.25)
Q3 = df[numerical_col].quantile(0.75)
IQR = Q3-Q1

LW = Q1-1.5*IQR
UW = Q3+1.5*IQR

# Outliers_count = df[(df[numerical_col]<LW) |(df[numerical_col]>UW)].sum()
# Outliers_percentage = Outliers_count / len(numrical_col)*100
stats= []

for i in  numerical_col:
    numerical_stats =OrderedDict({

        "feature":i,
        "minimum":df[i].min(),
        "maximum":df[i].max(),
        "mean":df[i].mean(),
        "median":df[i].median(),
        "standart deviation":df[i].std(),
        "skewness":df[i].skew(),
        "kurtosis":df[i].kurt(),
        "Q1":Q1[i],
        "Q3":Q3[i],
        "IQR":IQR[i],
        # "Outliers_count":Outliers_count[i],
        # "Outliers_percentage":Outliers_percentage[i]

    })
    stats.append(numerical_stats)
    report=pd.DataFrame(stats)
    print(report)



# data_preprocessing
  #1) drop the duplicates
  #2) split the X and y i.e input feature and target
  #3) split the train test split
  #4) use scaler ( pipeline)
  #5) use the SMOTE 

df = df.drop_duplicates()
X =df.drop(columns=[' loan_status','loan_id'])
y =df[' loan_status']

X_train , X_test,y_train,y_test = train_test_split(X,y,
                                                test_size=0.3,
                                                random_state=3)

numerical_col = X.select_dtypes(exclude='object').columns
categorical_col = X.select_dtypes(include='object').columns

num_pipe = Pipeline(steps=[
    ("imputer",SimpleImputer(strategy="median")),
    ("scaler",RobustScaler())
])

cat_pipe =Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("Encoder",OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num",num_pipe,numerical_col),
    ("cat",cat_pipe,categorical_col)
])
X_train,y_train = preprocessor.fit_transform(X_train,y_train)
X_test = preprocessor.transform(X_test)



model_building=({
    "DecisionTree":DecisionTreeClassifier(),
    "RandomForestClassifier": RandomForestClassifier(),
    "AdaBoostClassifier":AdaBoostClassifier(),
    "GradientBoostingClassifier":GradientBoostingClassifier(),
    "ExtraTreesClassifier":ExtraTreesClassifier(),
    "XGBClassifier":XGBClassifier(),
    "Kneighbors":KNeighborsClassifier(),
    "SVM":SVC()
})

for model_name , model in model_building.items():
    model.fit(X_train,y_train)
    y_pred = model.predict(X_test)
    
    print(f"model name:{model}")
    print(f"classification report:{classification_report(y_test,y_pred)}")
                                                       

   feature  minimum  maximum    mean  median  standart deviation  skewness  \
0  loan_id        1     4269  2135.0  2135.0         1232.498479       0.0   

   kurtosis      Q1      Q3     IQR  
0      -1.2  1068.0  3202.0  2134.0  
             feature  minimum  maximum         mean  median  \
0            loan_id        1     4269  2135.000000  2135.0   
1   no_of_dependents        0        5     2.498712     3.0   

   standart deviation  skewness  kurtosis      Q1      Q3     IQR  
0         1232.498479  0.000000 -1.200000  1068.0  3202.0  2134.0  
1            1.695910 -0.017971 -1.256992     1.0     4.0     3.0  
             feature  minimum  maximum          mean     median  \
0            loan_id        1     4269  2.135000e+03     2135.0   
1   no_of_dependents        0        5  2.498712e+00        3.0   
2       income_annum   200000  9900000  5.059124e+06  5100000.0   

   standart deviation  skewness  kurtosis         Q1         Q3        IQR  
0        1.232498e+03  0.00

ValueError: Invalid classes inferred from unique values of `y`.  Expected: [0 1], got [' Approved' ' Rejected']

In [34]:
df.columns

Index(['loan_id', ' no_of_dependents', ' education', ' self_employed',
       ' income_annum', ' loan_amount', ' loan_term', ' cibil_score',
       ' residential_assets_value', ' commercial_assets_value',
       ' luxury_assets_value', ' bank_asset_value', ' loan_status'],
      dtype='str')